In [ ]:
import pandas as pd
import numpy as np

# Let's create a realistic mock dataset representing 3 Best Buy stores and 3 tech categories
np.random.seed(42)
dates = pd.date_range(start="2025-01-01", end="2026-06-01", freq="D")
stores = [102, 105, 110] # Mock Store Numbers
categories = ['Computing', 'Mobile', 'Home Theater']

data_rows = []
for date in dates:
    for store in stores:
        for cat in categories:
            # Generate realistic daily sales quantities with some random variation
            base_sales = 15 if cat == 'Computing' else (10 if cat == 'Mobile' else 5)
            sales = base_sales + np.random.randint(-3, 5)
            # Simulate a seasonal bump in Nov/Dec for holidays
            if date.month in [11, 12]:
                sales += np.random.randint(5, 15)

            data_rows.append([date, store, cat, max(0, sales)])

# Turn it into a structured data table (DataFrame)
df = pd.DataFrame(data_rows, columns=['Date', 'StoreID', 'Category', 'Units_Sold'])
print("--- Dataset Successfully Created! ---")
print(df.head(10)) # Displays the first 10 rows of your data

In [ ]:
import matplotlib.pyplot as plt

# Group the data by Category to see total units sold
category_sales = df.groupby('Category')['Units_Sold'].sum().reset_index()

# Create a clean visual chart
plt.figure(figsize=(8, 5))
plt.bar(category_sales['Category'], category_sales['Units_Sold'], color=['#0046be', '#fff200', '#333333']) # Best Buy brand colors
plt.title('Total Units Sold by Product Category')
plt.xlabel('Department')
plt.ylabel('Total Units')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. Feature Engineering: Convert dates into numbers the AI can learn from
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

# 2. Convert text categories into numeric codes (Computing=0, Home Theater=1, Mobile=2)
df['Category_Code'] = df['Category'].astype('category').cat.codes

# 3. Define our Inputs (X) and what we want to predict (y)
X = df[['StoreID', 'Category_Code', 'Month', 'DayOfWeek']]
y = df['Units_Sold']

# 4. Split data: 80% to train the AI, 20% to test if its predictions are accurate
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Initialize and train the Random Forest Regressor (The ML Model)
print("Training the predictive model... Please wait.")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Evaluate how close the AI's guesses are to reality
predictions = model.predict(X_test)
error = mean_absolute_error(y_test, predictions)

print("--- Model Training Complete! ---")
print(f"On average, the AI's predictions are accurate within {error:.2f} units of actual sales.")

In [ ]:
# 1. Create a mock "Future Forecast" for tomorrow at Best Buy Store 102
tomorrow_data = pd.DataFrame({
    'StoreID': [102, 102, 102],
    'Category_Code': [0, 1, 2], # Computing, Home Theater, Mobile
    'Month': [6, 6, 6],         # June
    'DayOfWeek': [0, 0, 0]      # Monday
})

# 2. Let the trained AI predict tomorrow's sales demand
tomorrow_data['Predicted_Sales'] = model.predict(tomorrow_data[['StoreID', 'Category_Code', 'Month', 'DayOfWeek']])

# 3. Bring back the readable category names for the dashboard
code_to_name = {0: 'Computing', '1': 'Home Theater', 2: 'Mobile'} # Map codes back to text
tomorrow_data['Department'] = tomorrow_data['Category_Code'].map({0: 'Computing', 1: 'Home Theater', 2: 'Mobile'})

# 4. Simulate actual store inventory currently sitting on the shelves
# We will purposefully make Computing stock low to trigger our system alert
tomorrow_data['Current_Inventory'] = [5, 40, 12]

# 5. Core IS Business Logic: Calculate Restock Urgency and Action Alerts
def trigger_restock_alert(row):
    # If what we have is less than what the AI expects us to sell, it's a critical stockout risk
    if row['Current_Inventory'] < row['Predicted_Sales']:
        return '🔴 CRITICAL: Stockout Risk'
    # If we have just enough to cover the AI prediction but it's tight, flag a warning
    elif row['Current_Inventory'] <= (row['Predicted_Sales'] * 1.5):
        return '🟡 WARNING: Low Stock'
    else:
        return '🟢 STABLE: Sufficient Stock'

# Apply the logic across the data
tomorrow_data['Alert_Status'] = tomorrow_data.apply(trigger_restock_alert, axis=1)

# Clean up the table view for the dashboard
dashboard_preview = tomorrow_data[['StoreID', 'Department', 'Current_Inventory', 'Predicted_Sales', 'Alert_Status']]
print("--- Inventory System Alert Logic Processed ---")
print(dashboard_preview)

In [ ]:
# Create the dashboard script file
with open('app.py', 'w') as f:
    f.write('''
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# 1. Setup Page Configuration
st.set_page_config(page_title="Best Buy Smart Inventory Dashboard", layout="wide")

st.title("🔵 Best Buy Emerging Tech: Smart Inventory & Restock Dashboard")
st.markdown("*Powered by Machine Learning Demand Forecasting*")
st.write("---")

# 2. Re-create the data and model inside the app for the live demo
np.random.seed(42)
dates = pd.date_range(start="2025-01-01", end="2026-06-01", freq="D")
stores = [102, 105, 110]
categories = ['Computing', 'Mobile', 'Home Theater']
data_rows = []
for date in dates:
    for store in stores:
        for cat in categories:
            base_sales = 15 if cat == 'Computing' else (10 if cat == 'Mobile' else 5)
            sales = base_sales + np.random.randint(-3, 5)
            if date.month in [11, 12]: sales += np.random.randint(5, 15)
            data_rows.append([date, store, cat, max(0, sales)])

df = pd.DataFrame(data_rows, columns=['Date', 'StoreID', 'Category', 'Units_Sold'])
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Category_Code'] = df['Category'].astype('category').cat.codes

X = df[['StoreID', 'Category_Code', 'Month', 'DayOfWeek']]
y = df['Units_Sold']
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X, y)

# 3. Sidebar Filters for Store Managers
st.sidebar.header("📍 Store Operations Filter")
selected_store = st.sidebar.selectbox("Select Best Buy Store Location ID:", stores)

# 4. Generate Tomorrow's Live Predictions for the Selected Store
tomorrow_df = pd.DataFrame({
    'StoreID': [selected_store, selected_store, selected_store],
    'Category_Code': [0, 1, 2],
    'Month': [6, 6, 6],
    'DayOfWeek': [0, 0, 0]
})
tomorrow_df['Predicted_Sales'] = model.predict(tomorrow_df[['StoreID', 'Category_Code', 'Month', 'DayOfWeek']])
tomorrow_df['Department'] = tomorrow_df['Category_Code'].map({0: 'Computing', 1: 'Home Theater', 2: 'Mobile'})

# Simulate dynamic inventory based on the store selected
if selected_store == 102: tomorrow_df['Current_Inventory'] = [5, 40, 12]
elif selected_store == 105: tomorrow_df['Current_Inventory'] = [18, 8, 15]
else: tomorrow_df['Current_Inventory'] = [2, 12, 35]

def get_alert(row):
    if row['Current_Inventory'] < row['Predicted_Sales']: return '🔴 CRITICAL: Stockout Risk'
    elif row['Current_Inventory'] <= (row['Predicted_Sales'] * 1.5): return '🟡 WARNING: Low Stock'
    return '🟢 STABLE: Sufficient Stock'

tomorrow_df['Alert_Status'] = tomorrow_df.apply(get_alert, axis=1)

# 5. Dashboard Metrics Layout
critical_count = sum(tomorrow_df['Alert_Status'].str.contains('CRITICAL'))
warning_count = sum(tomorrow_df['Alert_Status'].str.contains('WARNING'))

col1, col2, col3 = st.columns(3)
col1.metric("Selected Store Location", f"Store #{selected_store}")
col2.metric("Critical Stockout Risks", critical_count, delta="- Actions Required", delta_color="inverse")
col3.metric("Low Stock Warnings", warning_count)

st.write("### 📋 Daily Restock Action Plan")
final_display = tomorrow_df[['Department', 'Current_Inventory', 'Predicted_Sales', 'Alert_Status']]
st.dataframe(final_display.style.set_properties(**{'text-align': 'center'}), use_container_width=True)
''')
print("Dashboard code successfully written to app.py!")

In [ ]:
# 1. Install Streamlit and localtunnel
!pip install streamlit --quiet
!npm install -g localtunnel --quiet

# 2. Print your public IP address (localtunnel will ask for this as a password)
import urllib
print("🔑 YOUR PASSWORD IS:", urllib.request.urlopen('https://ident.me').read().decode('utf8'))

# 3. Expose port 8501 and launch the live web link
print("👉 CLICK THE LINK BELOW, PASTE THE PASSWORD ABOVE, AND CLICK SUBMIT!")
!lt --port 8501 & streamlit run app.py --client.browser.gatherUsageStats=false --server.enableCORS=false &> /dev/null &

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙🔑 YOUR PASSWORD IS: 136.118.219.124
👉 CLICK THE LINK BELOW, PASTE THE PASSWORD ABOVE, AND CLICK SUBMIT!
your url is: https://petite-tables-run.loca.lt
